# ImageToVideo_Wan22_Colab — một ảnh → video, Wan2.2 14B trên T4

Port từ `wan2_2/wan22_Lightx2v.ipynb` của [Isi-dev](https://github.com/Isi-dev/Google-Colab_Notebooks) vào hợp đồng của cụm Colab (`/generate/<type>`, callback đẩy kết quả về api, cloudflared).

## v4: thời gian đi đâu, và ba cần gạt mới

Bóc tách job v3 (81 khung 480×848, **974,2 s**):

| bước | giây |
|---|---|
| mã hoá prompt | 28,8 |
| mã hoá ảnh | 17,6 |
| nạp model cao | 1,9 |
| **denoise cao (3 bước)** | **432,9** |
| **denoise thấp (3 bước)** | **438,9** |
| giải mã | 32,4 |
| ghi video | 21,0 |

**Denoise chiếm 872 s = 90%**, tức **~145 giây mỗi bước**. Không có chỗ nào lãng phí để tối ưu — muốn nhanh chỉ còn cách **làm ít việc đi**: ít bước, ít khung, ít pixel. Thứ duy nhất "miễn phí" là 28,8 s mã hoá prompt.

v4 thêm:

1. **Cache prompt embeds** — bỏ 28,8 s từ job thứ hai trở đi. An toàn vì ở Wan2.2 ảnh **không** đi qua text encoder (model không có nhánh cross-attention ảnh, đã chứng minh ở mục v3).
2. **`buoc_cao` thành tham số** — trước đây công thức `min(3, steps-1)` làm `steps=4` ra **3 cao + 1 thấp** chứ không phải 2+2 như người gọi tưởng. Một cái bẫy im lặng.
3. **`lightx2v_cao` / `lightx2v_thap` thành tham số** — để thử giả thuyết *"cường độ 3.0 trên model high-noise bóp chết chuyển động"* mà không phải up bản mới mỗi lần.

**Kết quả v3 (đã đo):** LoRA `t2v` (nạp đủ 100%) và `i2v` (mất 16% key) chạy **cùng phiên cùng seed** → hai video khác nhau thật (lệch 20,0/255) **nhưng cả hai vẫn là camera đẩy vào, nhân vật ngồi yên**. Sửa LoRA đúng về kỹ thuật nhưng **không phải** nguyên nhân.

Cũng đo được: `v3-i2v` trùng **từng pixel** với `v2-j1` (0,0000/255 trên 81 khung) dù khác phiên, khác bản notebook, khác 23 giây chạy — **pipeline tất định tuyệt đối**, nên chỉ so sánh **thời gian** mới cần cùng phiên.

## v3: LoRA đang dùng **lệch kiến trúc** — đã sửa

Người dùng báo hàng loạt `WARNING:root:lora key not loaded: ...cross_attn.k_img...`. Đọc thẳng danh sách tensor của cả hai file:

| | tensor | chứa `_img` |
|---|---|---|
| model `wan2.2_i2v_high_noise_14B_Q4_K_M.gguf` | `cross_attn` chỉ có q/k/v/o/norm_q/norm_k | **0** |
| LoRA v1/v2 dùng — `lightx2v_I2V_14B_480p` (của Wan **2.1** I2V) | 1.749 | **280 = 16,0%** |
| LoRA khớp — `lightx2v_T2V_..._v2_rank32` | 1.459 | **0** |

Wan2.2 đưa ảnh vào bằng **ghép latent**, không có nhánh cross-attention ảnh như Wan2.1 — nên 280 key `_img` của LoRA không có chỗ bám và **16% LoRA chưng cất bị vứt lặng lẽ mỗi lần chạy**.

Bản T2V chính là URL tác giả gốc để **comment ngay bên dưới**, và là bản notebook Wan2.2 Animate của họ dùng.

v3 **tải cả hai** và cho chọn bằng tham số job `lora` (`'t2v'` mặc định | `'i2v'` như cũ). Không đổi hẳn, vì đã đo được **tốc độ GPU Colab lệch 12% giữa các phiên** (854 s ngày 15/09 vs ~955 s ngày 16/09, cùng tham số) — A/B ở hai phiên khác nhau là so nhầm vào máy chủ chứ không phải vào LoRA.

## Đã đo (v1)

Hai job trên T4: **854 s và 855 s** cho 81 khung 480×848 (tác giả gốc ghi ~13 phút — khớp).

**Bản đánh giá "không được" trước đây tạm thu hồi**: nó chưa chứng minh model kém, chỉ chứng minh phép thử hỏng. Hai lý do:

1. **`negative_prompt` hoàn toàn vô hiệu.** Job thường và job "âm" ra file **trùng md5** (`c9947d09…`). `cfg=1` làm ComfyUI bỏ hẳn nhánh uncond. Mà phần lớn negative prompt tiếng Trung của tác giả là **chống tĩnh** (`静态`, `静止`, `静止不动的画面`) — cơ chế chống tĩnh duy nhất đã chết từ đầu. **Không nâng cfg được**: lightx2v là LoRA *cfg-step-distill*, chạy đúng ở cfg=1; nâng cfg là phá chưng cất.
2. **Prompt mặc định v1 tự xin người đứng yên** — `subtle shifts`, `small … movements`, `stable camera`. Đo khung: chênh lệch khung liên tiếp tb **8,16**, khung đầu vs cuối **56,10** → ảnh không hề tĩnh, nhưng đó là **camera zoom mạnh còn nhân vật giữ nguyên tư thế**. Ngược cả hai vế.

**v2 đổi prompt mặc định sang hướng chuyển động rõ ràng**, giữ nguyên văn bản cũ ở `MAC_DINH['chuyen_dong_nhe']` để so ngược bằng một lần gọi.

**Đã đo v2 (16/09)** — 5 job, 81 khung 480×848: **960,0 / 952,1 / 950,9 / 953,5 / 958,1 giây**. Trung bình **~955 s**, dao động chỉ **1%** — nội dung ảnh gần như không ảnh hưởng thời gian.

Kết quả chất lượng: **đổi prompt KHÔNG cứu được** — nhân vật vẫn gần như đứng yên, chỉ có camera đẩy vào. Nhưng phép thử đó chạy trên nền LoRA lệch kiến trúc (xem mục v3), nên chưa phải kết luận cuối về model.

Vì sao chạy được trên T4 dù model 14B huấn luyện bf16: GGUF Q4 ép 14B xuống ~9 GB, ComfyUI tự lo dtype (fp16 có upcast ở chỗ nhạy), LoRA lightx2v rút 20 bước xuống 6. Hai model (high/low noise) nạp lần lượt, không cùng lúc.

Chỗ hay chết: **RAM 12,7 GB** của T4, không phải VRAM. `/laststats` ghi mốc RAM/VRAM từng bước để thấy chết ở đâu.

Cell cuối giữ chạy; đợi dòng `PUBLIC_URL=` rồi gọi `GET /version` trước.

In [ ]:
# ============================================================================
#  ImageToVideo_Wan22_Colab.ipynb
#  backend = wan22_i2v   (1 notebook = 1 phuong an, KHONG gop nhieu model 1 file)
#  Port tu wan2_2/wan22_Lightx2v.ipynb  (repo https://github.com/Isi-dev/Google-Colab_Notebooks @ ae58404)
# ============================================================================
NOTEBOOK_NAME    = 'ImageToVideo_Wan22_Colab.ipynb'
NOTEBOOK_VERSION = 'v4'        # <-- BUMP moi khi sua notebook + cap nhat CHANGELOG
TEST_CASE        = 'W4-81f-480p-Q4-6b-loraT2V'    #@param {type:"string"}
BACKEND          = 'wan22_i2v'

# Tên type trong ai_types của liveportrait-auto/config.json. api dựng đường dẫn
# THẲNG từ tên này ('/generate/<type>'), nên nó phải khớp từng ký tự.
AI_TYPE = 'image_to_video_wan22'
AI_TYPE_ALIAS = ['wan22_i2v', 'image_to_video_wan']

# Cong bo qua GET /version -> khong bao gio nham ban nao / model nao / case nao
MODELS = [
    {
        "name": "Wan2.2 I2V A14B (high noise + low noise), GGUF Q4_K_M",
        "code": "https://github.com/Wan-Video/Wan2.2",
        "weights": "hf:Isi99999/Wan2.2BasedModels (Q4_K_M 9,65 GB x2) + umt5-xxl fp8 6,7 GB + wan_2.1_vae + CA HAI LoRA lightx2v rank32: T2V v2 (319 MB, KHOP kien truc Wan2.2) va I2V 480p (373 MB, ban v1/v2 dung, LECH 16% key)",
        "license": "Apache-2.0 (DUNG THUONG MAI DUOC)",
        "size": "~26 GB tai ve. Hai model nap LAN LUOT (cao roi thap), khong cung luc."
    }
]

BACKEND_PARAMS = ('anh_url', 'chuyen_dong', 'negative_prompt', 'so_frame', 'rong', 'cao', 'steps', 'guidance', 'seed', 'fps', 'lora', 'buoc_cao', 'lightx2v_cao', 'lightx2v_thap')

# Bộ mặc định = ĐÚNG bộ tác giả gốc đã đo trên T4 (xem cell 0). Đổi là phải đo lại.
MAC_DINH = {
    "so_frame": 81,
    "fps": 16,
    "ngan_sach_pixel": 399360,
    "steps": 6,
    "buoc_cao": 3,
    "guidance": 1.0,
    "sampler": "euler",
    "scheduler": "simple",
    "shift": 8.0,
    "lightx2v_cao": 3.0,
    "lightx2v_thap": 1.5,
    # 't2v' = lightx2v_T2V_..._v2 (1.459 tensor, 0 key '_img') — KHOP voi Wan2.2,
    #         nap het, khong con dong WARNING 'lora key not loaded'.
    # 'i2v' = lightx2v_I2V_14B_480p (1.749 tensor, 280 key '_img' = 16% bi vut) —
    #         dung y ban v1/v2 va y tac gia goc, giu lai de so nguoc.
    "lora": "t2v",
    "sage_attention": True,
    # v1 xin CU DONG NHO ("subtle", "small") va "stable camera" — do lai tren job that
    # thay dung cai minh xin: nguoi gan nhu khong doi tu the. Ma chinh cai co che chong
    # tinh (negative prompt "静态/静止") thi lai chet vi cfg=1. Nen v2 xin THANG chuyen
    # dong ro rang o prompt DUONG, cho giong prompt vi du cua tac gia goc
    # ("The beautiful woman walks forward and smiles as the camera pulls out").
    "chuyen_dong": "The person in the image moves clearly and naturally: turns the head, shifts body weight, moves the arms and hands with visible motion, blinks and changes facial expression. Photorealistic, realistic skin and lighting, consistent identity and clothing.",
    # Giu lai NGUYEN VAN prompt cua v1 de so nguoc duoc bang mot lan goi
    # (chuyen_dong = gia tri nay), khong phai doi notebook.
    "chuyen_dong_nhe": "The person in the image moves naturally and realistically: subtle shifts of posture, natural breathing, small realistic hand and head movements, natural blinking and facial expression. Photorealistic, realistic skin and lighting, consistent identity and clothing, stable camera.",
    "negative_prompt": "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走"
}

CHANGELOG = {
    "v4": "BOC TACH thoi gian that cua job v3 (81 khung 480x848, 974,2 s): ma_hoa_prompt 28,8 / "
          "ma_hoa_anh 17,6 / nap_model 1,9 / denoise_cao 432,9 (3 buoc) / denoise_thap 438,9 "
          "(3 buoc) / giai_ma 32,4 / ghi_video 21,0. DENOISE CHIEM 872 s = 90%, tuc ~145 s MOI "
          "BUOC. Ket luan thang: KHONG co cho nao lang phi de toi uu — muon nhanh chi co cach "
          "lam it viec di (it buoc, it khung, it pixel). Thu duy nhat mien phi la 28,8 s "
          "ma_hoa_prompt. "
          "v4 them ba can gat: (1) CACHE prompt embeds theo (prompt, negative) — bo 28,8 s tu "
          "job thu hai tro di; an toan vi o Wan2.2 anh KHONG di qua duong text encoder (model "
          "khong co nhanh cross-attention anh, da chung minh o v3). (2) `buoc_cao` thanh tham so "
          "job: truoc day cong thuc min(MAC_DINH[buoc_cao]=3, steps-1) lam steps=4 ra 3 cao + 1 "
          "thap chu khong phai 2+2 nhu nguoi goi tuong — mot cai bay im lang. (3) `lightx2v_cao` "
          "va `lightx2v_thap` thanh tham so job de thu gia thuyet 'cuong do 3.0 tren model "
          "high-noise bop chet chuyen dong' ma khong phai up ban moi moi lan. "
          "Boi canh do v3: LoRA t2v (nap du 100%, het sach WARNING) va i2v (mat 16% key) chay "
          "CUNG PHIEN cung seed -> hai video khac nhau that (lech 20,0/255) NHUNG ca hai VAN la "
          "camera day vao, nhan vat ngoi yen. Sua LoRA dung ve ky thuat nhung KHONG phai nguyen "
          "nhan nhan vat khong cu dong. Cung do duoc: v3-i2v trung TUNG PIXEL voi v2-j1 "
          "(0,0000/255 tren 81 khung) du khac phien, khac ban notebook, khac 23 giay chay — "
          "pipeline tat dinh tuyet doi, nen chi so sanh THOI GIAN moi can cung phien. "
          "CHUA DO LAI v4.",
    "v3": "Sua mot LOI THAT phat hien tu dong WARNING 'lora key not loaded' nguoi dung bao. "
          "DOC THANG danh sach tensor cua ca hai file, khong doan: "
          "(1) wan2.2_i2v_high_noise_14B_Q4_K_M.gguf co blocks.N.cross_attn CHI gom "
          "q/k/v/o/norm_q/norm_k; tim 'k_img', 'v_img', 'norm_k_img', 'img_emb' -> 0 KET QUA. "
          "Wan2.2 dua anh vao bang GHEP LATENT, khong co nhanh cross-attention anh nhu Wan2.1. "
          "(2) LoRA ban v1/v2 nap la lightx2v_I2V_14B_480p (cua Wan 2.1 I2V): 1.749 tensor, "
          "280 cai chua '_img' = 16,0% => 16% LoRA chung cat bi VUT LANG LE moi lan chay. "
          "(3) Ban khop la lightx2v_T2V_14B_cfg_step_distill_v2_rank32: 1.459 tensor, 0 cai '_img' "
          "— dung la URL tac gia goc de COMMENT ngay ben duoi, va la ban notebook Wan2.2 Animate "
          "cua chinh ho dung. v3 TAI CA HAI (319 + 373 MB) va cho chon bang tham so job `lora` "
          "('t2v' mac dinh | 'i2v' nhu cu). "
          "Vi sao khong doi han ma phai chon duoc: da do toc do GPU Colab LECH 12% GIUA CAC PHIEN "
          "(854 s ngay 15/09 vs ~955 s ngay 16/09, cung tham so cung anh), nen A/B o hai phien "
          "khac nhau la so nham vao may chu chu khong phai vao LoRA — phai chay hai ban trong "
          "CUNG MOT phien. "
          "Do that ngay 16/09 truoc khi sua, 5 job v2 (phien sau): 960,0 / 952,1 / 950,9 / 953,5 / "
          "958,1 giay cho 81 khung 480x848 — trung binh ~955 s, dao dong chi 1%, noi dung anh "
          "gan nhu khong anh huong thoi gian. "
          "CHUA DO LAI v3. Cau hoi v3 phai tra loi: 16% LoRA bi mat co phai ly do nhan vat "
          "khong cu dong (chi co camera troi) hay khong.",
    "v2": "DO THAT v1 tren T4 ngay 15/09/2026, hai job 854 s va 855 s cho 81 khung 480x848. "
          "Ban danh gia truoc day ket luan 'khong duoc'. Doc lai hai ket qua do thi thay "
          "phep thu CHUA CONG BANG voi model, vi hai le. "
          "(a) NEGATIVE PROMPT HOAN TOAN VO HIEU — chung minh duoc chu khong phai suy doan: "
          "job thuong va job 'am' ra file TRUNG md5 (c9947d09d0e83ba372600d5fa90cfb7f); "
          "LTXV13B cung vay (18bf00e4cc7abc287bf46e28548c2d16). Ly do: cfg=1 lam ComfyUI bo "
          "han nhanh uncond. Ma phan lon negative prompt tieng Trung cua tac gia la CHONG TINH "
          "(静态, 静止, 静止不动的画面) — tuc co che chong tinh duy nhat da chet tu dau. "
          "KHONG sua duoc bang cach nang cfg: lightx2v la LoRA cfg-step-distill, nang cfg la "
          "pha chung cat. "
          "(b) PROMPT MAC DINH v1 TU XIN NGUOI DUNG YEN: 'subtle shifts', 'small ... movements', "
          "'stable camera'. Do khung: chenh lech khung lien tiep tb 8,16 / khung dau vs cuoi 56,10 "
          "— tuc anh KHONG tinh chut nao, nhung la CAMERA ZOOM MANH con nhan vat giu nguyen tu the. "
          "Dung nguoc cai minh xin o ca hai ve. "
          "Da doi chieu lai toan bo tham so port: lightx2v 3.0/1.5, 6 buoc (3 buoc cao), cfg 1, "
          "shift 8, sage attention, va DUNG ban LoRA I2V (lightx2v_I2V_14B_480p...) chu khong "
          "phai ban T2V — port trung thanh voi tac gia, khac biet nam o prompt. "
          "v2 sua hai thu: (1) chuyen_dong mac dinh doi sang huong CHUYEN DONG RO RANG, giu nguyen "
          "van ban cu o MAC_DINH['chuyen_dong_nhe'] de so nguoc bang mot lan goi; (2) in canh bao "
          "khi cfg<=1 va ghi LAST_STATS['negative_co_tac_dung'] de khong ai do lai phep thu negative. "
          "KHONG doi: quant, LoRA, so buoc, kich thuoc — v2 chi doi prompt nen thoi gian van ~854 s. "
          "CHUA DO LAI v2. Ket luan v1 'khong duoc' TAM THU HOI: no chua chung minh model kem, "
          "chi chung minh phep thu hong.",
    "v1": "Port tu Isi-dev wan2_2/wan22_Lightx2v.ipynb vao hop dong cua cum (FastAPI + /generate/<type> + callback day ve api + cloudflared). GIU NGUYEN bo tham so tac gia goc da do tren T4: Q4_K_M, lightx2v (cao 3.0 / thap 1.5), 6 buoc (3 buoc model cao), cfg 1, euler/simple, shift 8, sage attention. Tac gia ghi: 5 giay 480p (81 khung) ~13 phut, 4,5 giay 720p ~35 phut — CHUA tu do lai. Khac tac gia: (1) kich thuoc suy tu ti le anh vao o ngan sach 832x480 (bai hoc LTX v4: cat giua lam mat nguoi), (2) khong tai 3 LoRA prompt_assist va clip_vision (tac gia khong dung), (3) chan video den/NaN truoc khi ghi (bai hoc Qwen), (4) sage attention hong thi bo qua chu khong chet, (5) seed mac dinh 42 de do lap lai duoc."
}

import shutil, subprocess


def _sh(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()
    except Exception:
        return ''


print(f'=== {NOTEBOOK_NAME} {NOTEBOOK_VERSION} | backend={BACKEND} | case={TEST_CASE} ===')
for m in MODELS:
    print(f"    model: {m['name']}  [{m['license']}]  {m['weights']}")
print('GPU:', _sh(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
      or '!!! CHUA BAT GPU: Runtime > Change runtime type > GPU')
print('GPU yeu cau:', 'T4 (16 GB) — tac gia goc do: 5s 480p (81 khung) ~13 phut voi Q4_K_M; 720p ~35 phut')
print('ffmpeg:', shutil.which('ffmpeg') or 'THIEU (cell cai dat se cai)')


=== ImageToVideo_Wan22_Colab.ipynb v4 | backend=wan22_i2v | case=W4-81f-480p-Q4-6b-loraT2V ===
    model: Wan2.2 I2V A14B (high noise + low noise), GGUF Q4_K_M  [Apache-2.0 (DUNG THUONG MAI DUOC)]  hf:Isi99999/Wan2.2BasedModels (Q4_K_M 9,65 GB x2) + umt5-xxl fp8 6,7 GB + wan_2.1_vae + CA HAI LoRA lightx2v rank32: T2V v2 (319 MB, KHOP kien truc Wan2.2) va I2V 480p (373 MB, ban v1/v2 dung, LECH 16% key)
GPU: Tesla T4, 15360 MiB
GPU yeu cau: T4 (16 GB) — tac gia goc do: 5s 480p (81 khung) ~13 phut voi Q4_K_M; 720p ~35 phut
ffmpeg: /usr/bin/ffmpeg


In [ ]:
import os, shutil, subprocess, sys, time

COMFY = '/content/ComfyUI'
_T0 = time.time()


def _run(cmd, cwd=None, check=True, quiet=True):
    """Chay lenh, in dong lenh truoc de log Colab doc duoc dang o buoc nao."""
    print('$', ' '.join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str),
                       capture_output=quiet, text=True)
    if check and r.returncode != 0:
        print(r.stdout[-3000:] if r.stdout else '')
        print(r.stderr[-3000:] if r.stderr else '')
        # Dua duoi stderr VAO thong diep loi: traceback Colab hien no ngay, khong phai
        # cuon len tim (lan dau vo, chi thay 'lenh loi (2)' ma khong biet vi sao).
        duoi = (r.stderr or r.stdout or '').strip()[-1200:]
        raise RuntimeError(f'lenh loi ({r.returncode}): {cmd}' + chr(10) + '--- stderr ---' + chr(10) + duoi)
    return r


def _pip(*pk):
    _run([sys.executable, '-m', 'pip', 'install', '-q', *pk])


def _clone(url, dst, branch=None):
    if os.path.isdir(os.path.join(dst, '.git')):
        print(f'   da co {dst}, bo qua clone')
        return
    cmd = ['git', 'clone', '-q', '--depth', '1']
    if branch:
        cmd += ['--branch', branch]
    _run(cmd + [url, dst])


def _aria(url, thu_muc, ten=None):
    """Tai bang aria2c 16 luong (cach tac gia goc). Co roi thi bo qua."""
    ten = ten or url.split('/')[-1].split('?')[0]
    os.makedirs(thu_muc, exist_ok=True)
    d = os.path.join(thu_muc, ten)
    if os.path.exists(d) and os.path.getsize(d) > 1e6:
        print(f'   da co {ten} ({os.path.getsize(d) / 1e9:.2f} GB)')
        return d
    t0 = time.time()
    _run(['aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M',
          '--summary-interval=0', '-d', thu_muc, '-o', ten, url])
    print(f'   {ten}: {os.path.getsize(d) / 1e9:.2f} GB trong {time.time() - t0:.0f}s')
    return d


_run(['apt-get', '-y', 'install', '-qq', 'aria2', 'ffmpeg'], check=False)

# cloudflared — cell server goi thang /usr/local/bin/cloudflared va KIEM LAI ngay sau khi
# cai (bai hoc tu notebook Qwen: cai xong, model san sang, roi vo o dong cuoi cung).
if not shutil.which('cloudflared'):
    _run('curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/'
         'releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared')
if not shutil.which('cloudflared'):
    raise RuntimeError('khong cai duoc cloudflared -- cell server se khong mo duoc tunnel')

# Torch GHIM dung ban tac gia goc dung. Colab doi torch moi lien tuc; ban khac la
# xformers/sageattention khong khop wheel. Cai lai mat vai phut nhung chac.
_pip('torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0')

_pip('torchsde', 'einops', 'diffusers', 'accelerate', 'xformers==0.0.29.post2',
     'triton==3.2.0', 'sageattention==1.0.6', 'av', 'spandrel', 'opencv-python',
     'imageio[ffmpeg]', 'psutil', 'fastapi', 'uvicorn', 'python-multipart', 'requests')

_clone('https://github.com/Isi-dev/ComfyUI', COMFY, branch='ComfyUI_v0.3.47')
_clone('https://github.com/Isi-dev/ComfyUI_GGUF.git', f'{COMFY}/custom_nodes/ComfyUI_GGUF')
_clone('https://github.com/Isi-dev/ComfyUI_KJNodes.git', f'{COMFY}/custom_nodes/ComfyUI_KJNodes',
       branch='kjnv1.1.3')
# KHONG cai ComfyUI/requirements.txt: no ghim comfyui-frontend-package / workflow-templates /
# embedded-docs (chi cho giao dien web, ta khong dung) va cac ban ghim do da bien khoi
# PyPI -> pip loi. Tac gia goc cung khong cai tep nay; goi loi `nodes` chi can bo duoi.
_pip('aiohttp', 'yarl', 'pyyaml', 'scipy', 'tqdm', 'psutil', 'safetensors', 'sentencepiece',
     'tokenizers', 'kornia', 'soundfile', 'pydantic', 'pydantic-settings')
_pip('-r', f'{COMFY}/custom_nodes/ComfyUI_GGUF/requirements.txt')
_pip('-r', f'{COMFY}/custom_nodes/ComfyUI_KJNodes/requirements.txt')

# Wan2.2 I2V A14B = HAI model (high noise + low noise), moi cai Q4_K_M ~9 GB.
QUANT = 'Q4_K_M'   # tac gia goc: T4 chi chay noi Q4_K_M; Q5/Q6/Q8 de danh cho L4+
DM = f'{COMFY}/models/diffusion_models'
DIT_CAO = _aria(f'https://huggingface.co/Isi99999/Wan2.2BasedModels/resolve/main/wan2.2_i2v_high_noise_14B_{QUANT}.gguf', DM)
DIT_THAP = _aria(f'https://huggingface.co/Isi99999/Wan2.2BasedModels/resolve/main/wan2.2_i2v_low_noise_14B_{QUANT}.gguf', DM)
TE = _aria('https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', f'{COMFY}/models/text_encoders')
VAE = _aria('https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors', f'{COMFY}/models/vae')
# LoRA lightx2v: chung cat rut so buoc 20 -> 6 (cfg 1). Day la thu lam T4 chay 13 phut
# thay vi 26. Tac gia ap no len CA HAI model, cuong do 3 (cao) va 1,5 (thap).
# TAI CA HAI de doi duoc ngay trong job. Khong the so hai ban o hai phien Colab
# khac nhau: da do duoc toc do GPU lech 12% giua cac phien (854 s vs ~955 s voi
# cung tham so), nen A/B khac phien la so nham vao may chu khong phai vao LoRA.
LORA_T2V = _aria('https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank32_bf16.safetensors', f'{COMFY}/models/loras')
LORA_I2V = _aria('https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/lightx2v_I2V_14B_480p_cfg_step_distill_rank32_bf16.safetensors', f'{COMFY}/models/loras')
LORA_LIGHT = LORA_T2V

print(f'>>> CAI DAT XONG ({time.time() - _T0:.0f}s).')


$ apt-get -y install -qq aria2 ffmpeg
$ curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
$ /usr/bin/python3 -m pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0
$ /usr/bin/python3 -m pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2 triton==3.2.0 sageattention==1.0.6 av spandrel opencv-python imageio[ffmpeg] psutil fastapi uvicorn python-multipart requests
$ git clone -q --depth 1 --branch ComfyUI_v0.3.47 https://github.com/Isi-dev/ComfyUI /content/ComfyUI
$ git clone -q --depth 1 https://github.com/Isi-dev/ComfyUI_GGUF.git /content/ComfyUI/custom_nodes/ComfyUI_GGUF
$ git clone -q --depth 1 --branch kjnv1.1.3 https://github.com/Isi-dev/ComfyUI_KJNodes.git /content/ComfyUI/custom_nodes/ComfyUI_KJNodes
$ /usr/bin/python3 -m pip install -q aiohttp yarl pyyaml scipy tqdm psutil safetensors sentencepiece tokenizers kornia soundfile 

In [ ]:
import glob, json, os, re, shutil, subprocess, time
from typing import Optional

import numpy as np
from PIL import Image, ImageOps
from pydantic import BaseModel

JOB_DIR = '/content/jobs'
RES_DIR = '/content/results'
WRK_DIR = '/content/work'
for d in (JOB_DIR, RES_DIR, WRK_DIR):
    os.makedirs(d, exist_ok=True)

LAST_STATS = {}


def _slug(s, default='case'):
    s = re.sub(r'[^A-Za-z0-9._-]+', '-', (s or '').strip()).strip('-.')
    return (s or default)[:48]


def _moc(ten):
    """Moc bo nho (RAM/VRAM) tai tung buoc. T4 hay chet o RAM 12,7 GB chu khong phai
    VRAM, nen phai thay duoc RAM tang o buoc nao."""
    try:
        import psutil
        ram = round(psutil.Process(os.getpid()).memory_info().rss / 1024 ** 3, 2)
    except Exception:
        ram = None
    try:
        import torch
        vram = round(torch.cuda.memory_allocated() / 1024 ** 3, 2) if torch.cuda.is_available() else None
    except Exception:
        vram = None
    LAST_STATS.setdefault('_bo_nho', {})[ten] = {'ram_gb': ram, 'vram_gb': vram, 't': round(time.time(), 1)}


def probe_video(path):
    out = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
         'stream=width,height,r_frame_rate,nb_frames', '-of', 'json', path],
        capture_output=True, text=True, check=True).stdout
    st = json.loads(out)['streams'][0]
    num, den = (st['r_frame_rate'].split('/') + ['1'])[:2]
    fps = float(num) / float(den or 1)
    return int(st['width']), int(st['height']), fps, int(st.get('nb_frames') or 0)


def encode_video(frames_dir, fps, out_path, tag=''):
    """PNG -> mp4 h264 yuv420p. Metadata comment mang du notebook + version + backend + case
    -> ffprobe file mp4 la biet ngay ban nao sinh ra no."""
    comment = (f'notebook={NOTEBOOK_NAME} version={NOTEBOOK_VERSION} backend={BACKEND} {tag}').strip()
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-framerate', f'{fps:.6f}', '-start_number', '0',
                    '-i', os.path.join(frames_dir, '%06d.png'),
                    '-c:v', 'libx264', '-preset', 'medium', '-crf', '18', '-pix_fmt', 'yuv420p',
                    '-metadata', f'comment={comment}', out_path], check=True)
    return out_path


def reencode_video(src, out_path, tag=''):
    comment = (f'notebook={NOTEBOOK_NAME} version={NOTEBOOK_VERSION} backend={BACKEND} {tag}').strip()
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', src, '-c:v', 'libx264', '-preset', 'medium',
                    '-crf', '18', '-pix_fmt', 'yuv420p', '-metadata', f'comment={comment}', out_path],
                   check=True)
    return out_path


def save_frames_u8(arr, out_dir):
    """arr: (N,H,W,3) uint8 -> 000000.png ..."""
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    for i in range(arr.shape[0]):
        Image.fromarray(arr[i]).save(os.path.join(out_dir, f'{i:06d}.png'))
    return out_dir


def doc_anh_rgb(path):
    """PNG RGBA (khuon mat / outfit trong repo la RGBA) -> ep len nen TRANG roi RGB.
    .convert('RGB') thang se bien vung trong suot thanh mau rac."""
    im = Image.open(path)
    im = ImageOps.exif_transpose(im)
    if im.mode in ('RGBA', 'LA') or (im.mode == 'P' and 'transparency' in im.info):
        im = im.convert('RGBA')
        bg = Image.new('RGBA', im.size, (255, 255, 255, 255))
        im = Image.alpha_composite(bg, im)
    return im.convert('RGB')


def _kich_thuoc_theo_anh(w0, h0, ngan_sach, boi=16, toi_thieu=256):
    """Giu ti le anh vao, tong pixel ~= ngan_sach, moi canh chia het cho `boi`.
    Bai hoc LTX v4: cat giua ve 704x512 lam anh doc mat 51% chieu cao (chi con khuon mat)."""
    ti_le = w0 / max(1, h0)
    h = (ngan_sach / ti_le) ** 0.5
    w = h * ti_le
    w = max(toi_thieu, int(round(w / boi)) * boi)
    h = max(toi_thieu, int(round(h / boi)) * boi)
    return w, h


def _khung_4n1(n, toi_thieu=5):
    """Wan: so khung phai la 4n+1. Lam tron XUONG va bao ro."""
    n = max(toi_thieu, int(n))
    return (n - 1) // 4 * 4 + 1


def _khung_8n1(n, toi_thieu=9):
    """LTX: so khung phai la 8n+1."""
    n = max(toi_thieu, int(n))
    return (n - 1) // 8 * 8 + 1


def _tai_ve(url, dst):
    from urllib.parse import urlsplit
    import requests
    sp = urlsplit(url)
    r = requests.get(url, stream=True, timeout=300,
                     headers={'User-Agent': 'Mozilla/5.0', 'Referer': f'{sp.scheme}://{sp.netloc}/'})
    r.raise_for_status()
    with open(dst, 'wb') as f:
        for chunk in r.iter_content(1 << 16):
            f.write(chunk)
    return dst


def _duoi(url, mac_dinh):
    from urllib.parse import urlparse
    e = os.path.splitext(urlparse(url or '').path or (url or ''))[1].lower()
    return e if e else mac_dinh


class Job(BaseModel):
    """Than JSON cua POST /generate/<type>. Ten truong giu giong LTX de app khong phai hoc lai."""
    anh_url: str
    test_case: str = ''
    # Cau ta chuyen dong. De trong = MAC_DINH['chuyen_dong'].
    chuyen_dong: str = ''
    negative_prompt: str = ''
    # None = de MAC_DINH quyet. Gui gia tri thi no THANG mac dinh.
    so_frame: Optional[int] = None
    rong: Optional[int] = None
    cao: Optional[int] = None
    steps: Optional[int] = None
    guidance: Optional[float] = None
    fps: Optional[int] = None
    lora: Optional[str] = None      # 't2v' (mac dinh, khop Wan2.2) | 'i2v' (nhu v1/v2)
    # So buoc danh cho model HIGH-NOISE. None = MAC_DINH (3). Voi steps=4 thi nen dat 2
    # de chia 2+2; de None se ra 3+1 vi cong thuc cu la min(buoc_cao, steps-1).
    buoc_cao: Optional[int] = None
    # Cuong do LoRA lightx2v tren tung model. Tac gia goc: cao 3.0 / thap 1.5.
    # high-noise la tang quyet dinh CHUYEN DONG TONG THE — ha rieng no xuong de thu
    # gia thuyet "3.0 bop chet chuyen dong".
    lightx2v_cao: Optional[float] = None
    lightx2v_thap: Optional[float] = None
    seed: int = 42
    job_id: Optional[str] = None
    # api tự đặt hai trường này, người gọi KHÔNG được gửi lên.
    callback_url: Optional[str] = None
    callback_token: Optional[str] = None


def chay_job(job, uid, out_path, progress=None):
    """Tai anh ve, ep ve RGB, roi giao cho run_backend. Tach rieng de test duoc
    khong can GPU (test thay run_backend bang ban gia)."""
    if progress:
        progress('tai_anh', 0, 1)
    goc = _tai_ve(job.anh_url, os.path.join(JOB_DIR, f'{uid}_a{_duoi(job.anh_url, ".png")}'))
    anh = os.path.join(JOB_DIR, f'{uid}_a.png')
    doc_anh_rgb(goc).save(anh)
    # rong/cao KHONG co trong MAC_DINH: None = suy tu ti le anh vao (run_backend lo).
    if job.lora is not None and job.lora.lower() not in ('t2v', 'i2v'):
        raise ValueError(f"lora phai la 't2v' hoac 'i2v', nhan {job.lora!r}")
    th = {k: (getattr(job, k) if getattr(job, k) is not None else MAC_DINH.get(k))
          for k in ('so_frame', 'rong', 'cao', 'steps', 'guidance', 'fps', 'lora',
                    'buoc_cao', 'lightx2v_cao', 'lightx2v_thap')}
    return run_backend(anh, out_path, test_case=job.test_case or TEST_CASE,
                       chuyen_dong=job.chuyen_dong or MAC_DINH['chuyen_dong'],
                       negative_prompt=job.negative_prompt or MAC_DINH['negative_prompt'],
                       seed=job.seed, progress=progress, **th)


In [ ]:
import gc, os, subprocess, sys, time
import numpy as np
import torch

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
sys.path.insert(0, COMFY)
os.chdir(COMFY)   # ComfyUI tim models/ input/ theo cwd o vai cho

import comfy.utils
from comfy import model_management

_TIEN_DO = {'cb': None, 'buoc': ''}


def _bat_tien_do(progress):
    """Noi thanh tien do cua ComfyUI (tung buoc denoise) vao progress cua job."""
    _TIEN_DO['cb'] = progress

    def hook(value, total, *a, **k):
        cb = _TIEN_DO['cb']
        if cb and total:
            cb(_TIEN_DO['buoc'], value, total)
    comfy.utils.set_progress_bar_global_hook(hook)


def _buoc(ten, progress, done=0, total=1):
    _TIEN_DO['buoc'] = ten
    if progress:
        progress(ten, done, total)
    _moc(ten)
    print(f'   [{BACKEND}] {ten} ...', flush=True)


def _don():
    try:
        model_management.unload_all_models()
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def _tensor_sang_u8(decoded):
    """(N,H,W,3) float [0,1] -> uint8. Kiem NaN/den o day: ComfyUI KHONG bao, cu tra
    tensor rac roi ta ghi ra video den ma job van 'done' (bai hoc Qwen)."""
    x = decoded.detach().float().cpu()
    if not torch.isfinite(x).all():
        raise RuntimeError('tensor giai ma co NaN/Inf — pipeline tran so (fp16 tren T4?)')
    x = (x.clamp(0, 1) * 255).round().to(torch.uint8).numpy()
    if x.std() < 0.5:
        raise RuntimeError(f'video ra RONG (do lech chuan {x.std():.3f}/255) — moi pixel nhu nhau')
    return x


def _kiem_san_sang():
    """Chay luc preload: import node + kiem file model. Nhanh, khong nap gi len GPU."""
    thieu = [p for p in _FILE_CAN if not os.path.exists(p)]
    if thieu:
        raise RuntimeError('thieu file model: ' + ', '.join(thieu))
    print(f'   [{BACKEND}] du {len(_FILE_CAN)} file model, GPU: '
          f'{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO"}')


from nodes import (CLIPLoader, CLIPTextEncode, VAEDecode, VAELoader, KSamplerAdvanced,
                   LoadImage, LoraLoaderModelOnly, ImageScale)
from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from custom_nodes.ComfyUI_KJNodes.nodes.model_optimization_nodes import PathchSageAttentionKJ
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_wan import WanImageToVideo

_FILE_CAN = [DIT_CAO, DIT_THAP, TE, VAE, LORA_T2V, LORA_I2V]
PRELOAD = [('kiem_san_sang', _kiem_san_sang)]


def _vao_input(anh, uid):
    """LoadImage cua ComfyUI doc tu <COMFY>/input theo ten file."""
    d = os.path.join(COMFY, 'input')
    os.makedirs(d, exist_ok=True)
    ten = f'{uid}.png'
    import shutil as _sh
    _sh.copyfile(anh, os.path.join(d, ten))
    return ten


# ── Cache prompt embeds (bai hoc LTX v6 / VACE v2) ────────────────────────────
# Do v3: ma_hoa_prompt ton 28,8 s moi job (nap umt5 fp8 6,7 GB roi encode). Ket qua
# chi phu thuoc (prompt, negative) — anh KHONG di qua duong nay o Wan2.2 (model
# khong co nhanh cross-attention anh; xem changelog v3). Nen cung mot cap cau thi
# cung mot tensor. Tra BAN SAO de node phia sau co sua cung khong hong kho.
_EMBEDS, _EMBEDS_TRAN, _EMBEDS_DEM = {}, 16, {'trung': 0, 'truot': 0}


def _sao_cond(cond):
    return [[x.clone() if hasattr(x, 'clone') else x, dict(d)] for x, d in cond]


def _ma_hoa_prompt(chuyen_dong, negative_prompt):
    khoa = (chuyen_dong, negative_prompt)
    if khoa in _EMBEDS:
        _EMBEDS_DEM['trung'] += 1
        pos, neg = _EMBEDS[khoa]
        print(f'   [{BACKEND}] prompt da co trong kho -> bo qua nap umt5 '
              f'(kho {len(_EMBEDS)}/{_EMBEDS_TRAN}, tiet kiem ~29 s)')
        return _sao_cond(pos), _sao_cond(neg)
    _EMBEDS_DEM['truot'] += 1
    clip = CLIPLoader().load_clip(os.path.basename(TE), 'wan', 'default')[0]
    pos = CLIPTextEncode().encode(clip, chuyen_dong)[0]
    neg = CLIPTextEncode().encode(clip, negative_prompt)[0]
    del clip
    _don()
    if len(_EMBEDS) >= _EMBEDS_TRAN:
        _EMBEDS.pop(next(iter(_EMBEDS)))
    _EMBEDS[khoa] = (pos, neg)
    return _sao_cond(pos), _sao_cond(neg)


def _ap_lora_va_va(model, cuong_do, sage, progress, tep_lora=None):
    """Cung mot chuoi va cho ca model cao lan thap: shift 8 -> lightx2v -> sage attention."""
    model = ModelSamplingSD3().patch(model, MAC_DINH['shift'])[0]
    model = LoraLoaderModelOnly().load_lora_model_only(
        model, os.path.basename(tep_lora or LORA_LIGHT), cuong_do)[0]
    if sage:
        try:
            model = PathchSageAttentionKJ().patch(model, 'auto')[0]
        except Exception as e:
            # T4 la sm75; sageattention co the khong ho tro. Khong chet, chi cham hon.
            print(f'   [{BACKEND}] sage attention KHONG dung duoc ({type(e).__name__}: {e}) -> bo qua')
            LAST_STATS['_sage_attention'] = f'bo qua: {e}'
    return model


def run_backend(anh, out_path, test_case='', so_frame=81, rong=None, cao=None, steps=6,
                guidance=1.0, seed=42, fps=16, chuyen_dong='', negative_prompt='',
                lora=None, buoc_cao=None, lightx2v_cao=None, lightx2v_thap=None,
                progress=None):
    """Port tu generate_video() cua wan22_Lightx2v.ipynb, bo phan upload/hien thi.

    Trinh tu tac gia (giu nguyen): ma hoa prompt (umt5 fp8) -> giai phong -> anh ->
    WanImageToVideo (VAE) -> model CAO 3 buoc dau (them nhieu) -> giai phong -> model
    THAP cac buoc con lai -> VAE decode -> video.
    """
    LAST_STATS.clear()
    t0 = time.time()
    _bat_tien_do(progress)
    uid = os.path.splitext(os.path.basename(out_path))[0].split('_')[0]

    from PIL import Image as _I
    w0, h0 = _I.open(anh).size
    if rong is None or cao is None:
        rong, cao = _kich_thuoc_theo_anh(w0, h0, MAC_DINH['ngan_sach_pixel'], boi=16)
    n = _khung_4n1(so_frame)
    if n != so_frame:
        print(f'   [{BACKEND}] so_frame {so_frame} -> {n} (Wan can 4n+1)')
    # Tran `steps - 1` giu cho model thap it nhat mot buoc. Truoc v4 tri nay cung la
    # MAC_DINH['buoc_cao']=3, nen steps=4 ra 3+1 chu khong phai 2+2 nhu nguoi goi tuong.
    buoc_cao = min(int(buoc_cao if buoc_cao is not None else MAC_DINH['buoc_cao']),
                   max(1, int(steps) - 1))
    _ten_lora = (lora or MAC_DINH['lora']).lower()
    _tep_lora = LORA_I2V if _ten_lora == 'i2v' else LORA_T2V
    _cd_cao = float(lightx2v_cao if lightx2v_cao is not None else MAC_DINH['lightx2v_cao'])
    _cd_thap = float(lightx2v_thap if lightx2v_thap is not None else MAC_DINH['lightx2v_thap'])
    LAST_STATS.update({'_case': test_case, 'anh_vao': [w0, h0], 'video_ra': [rong, cao, n, fps],
                       'steps': steps, 'buoc_cao': buoc_cao, 'cfg': guidance, 'seed': seed,
                       'prompt': chuyen_dong[:200], 'lora': _ten_lora,
                       'lora_tep': os.path.basename(_tep_lora),
                       'lightx2v_cao': _cd_cao, 'lightx2v_thap': _cd_thap})
    print(f'   [{BACKEND}] LoRA {_ten_lora}: {os.path.basename(_tep_lora)}'
          + ('   (LECH kien truc — 16% key ._img bi vut, se thay WARNING)'
             if _ten_lora == 'i2v' else '   (khop kien truc Wan2.2)'))
    print(f'   [{BACKEND}] anh {w0}x{h0} -> video {rong}x{cao} x {n} khung @ {fps} fps, '
          f'{steps} buoc (cao {buoc_cao} + thap {steps - buoc_cao}), '
          f'lightx2v {_cd_cao}/{_cd_thap}, cfg {guidance}, seed {seed}')
    # cfg=1 => ComfyUI bo hoan toan nhanh uncond => negative_prompt khong bao gio chay.
    # Da chung minh: hai job v1 chi khac negative_prompt ra file TRUNG md5
    # (c9947d09d0e83ba372600d5fa90cfb7f). In canh bao o day de khong ai do lai mot
    # phep thu da biet ket qua.
    if float(guidance) <= 1.0:
        print(f'   [{BACKEND}] CHU Y cfg={guidance}: negative_prompt KHONG co tac dung. '
              'Khong nang cfg duoc — lightx2v la LoRA cfg-step-distill, chay dung o cfg=1; '
              'nang cfg la pha chung cat. Muon doi ket qua thi doi chuyen_dong (prompt duong).')
    LAST_STATS['negative_co_tac_dung'] = float(guidance) > 1.0

    with torch.inference_mode():
        _buoc('ma_hoa_prompt', progress)
        pos, neg = _ma_hoa_prompt(chuyen_dong, negative_prompt)
        LAST_STATS['_embeds'] = {'so_muc': len(_EMBEDS), **_EMBEDS_DEM}

        _buoc('ma_hoa_anh', progress)
        img = LoadImage().load_image(_vao_input(anh, uid))[0]
        img = ImageScale().upscale(img, 'lanczos', rong, cao, 'disabled')[0]
        vae = VAELoader().load_vae(os.path.basename(VAE))[0]
        pos_o, neg_o, latent = WanImageToVideo().encode(pos, neg, vae, rong, cao, n, 1, img, None)

        _buoc('nap_model_cao', progress)
        model = UnetLoaderGGUF().load_unet(os.path.basename(DIT_CAO))[0]
        model = _ap_lora_va_va(model, _cd_cao, MAC_DINH['sage_attention'], progress, _tep_lora)
        _buoc('denoise_cao', progress)
        ks = KSamplerAdvanced()
        sampled = ks.sample(model=model, add_noise='enable', noise_seed=seed, steps=steps,
                            cfg=guidance, sampler_name=MAC_DINH['sampler'],
                            scheduler=MAC_DINH['scheduler'], positive=pos_o, negative=neg_o,
                            latent_image=latent, start_at_step=0, end_at_step=buoc_cao,
                            return_with_leftover_noise='enable')[0]
        del model
        _don()

        _buoc('nap_model_thap', progress)
        model = UnetLoaderGGUF().load_unet(os.path.basename(DIT_THAP))[0]
        model = _ap_lora_va_va(model, _cd_thap, MAC_DINH['sage_attention'], progress, _tep_lora)
        _buoc('denoise_thap', progress)
        sampled = ks.sample(model=model, add_noise='disable', noise_seed=seed, steps=steps,
                            cfg=guidance, sampler_name=MAC_DINH['sampler'],
                            scheduler=MAC_DINH['scheduler'], positive=pos_o, negative=neg_o,
                            latent_image=sampled, start_at_step=buoc_cao, end_at_step=10000,
                            return_with_leftover_noise='disable')[0]
        del model
        _don()

        _buoc('giai_ma', progress)
        decoded = VAEDecode().decode(vae, sampled)[0]
        del vae, sampled
        _don()

    _buoc('ghi_video', progress)
    u8 = _tensor_sang_u8(decoded)
    del decoded
    khung = save_frames_u8(u8, os.path.join(WRK_DIR, uid))
    encode_video(khung, fps, out_path, tag=f'case={_slug(test_case)} seed={seed}')
    _don()
    LAST_STATS['_thoi_gian_s'] = round(time.time() - t0, 1)
    _moc('xong')
    print(f'   [{BACKEND}] xong {n} khung trong {time.time() - t0:.0f}s -> {out_path}')
    return out_path


In [ ]:
import os, re, socket, subprocess, threading, time, traceback, uuid
from typing import Optional

import requests
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse, HTMLResponse

PORT = 8000
STATE = {'models_ready': False}
app = FastAPI(title=f'{NOTEBOOK_NAME} [{BACKEND}] {NOTEBOOK_VERSION}')


# ============================================================ job store (async)
# Tra job_id ngay roi render o thread nen -> tranh Cloudflare 524 khi job chay vai chuc phut.
JOBS, JOBS_LOCK, RUN_LOCK, JOB_TTL = {}, threading.Lock(), threading.Lock(), 6 * 3600


def _prune_locked():
    now = time.time()
    for k in [k for k, v in JOBS.items() if now - v.get('created', now) > JOB_TTL]:
        v = JOBS.pop(k, None)
        try:
            if v and v.get('result') and os.path.exists(v['result']):
                os.remove(v['result'])
        except Exception:
            pass


def _new_job(jid=None, callback_url=None, callback_token=None):
    jid = jid or uuid.uuid4().hex
    with JOBS_LOCK:
        _prune_locked()
        JOBS[jid] = {'status': 'processing', 'stage': 'queued', 'progress': 0.0,
                     'result': None, 'filename': None, 'error': None, 'created': time.time(),
                     'callback_url': callback_url, 'callback_token': callback_token,
                     'da_day_ve': None}
    return jid


def _progress_cb(jid):
    def cb(stage, done, total):
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid]['stage'] = stage
                JOBS[jid]['progress'] = round(done / max(1, total), 4)
    return cb


# ── Colab tự đẩy kết quả về api ─────────────────────────────────────────────
# Đo trên chính hạ tầng này: quick tunnel cloudflared cho chiều api-KÉO khoảng
# 0,11 MB/s, chiều Colab-ĐẨY khoảng 8,5 MB/s — chênh khoảng 85 lần.
# Không có callback thì KHÔNG làm gì: api kéo qua GET /jobs/{id}/result như cũ.
def day_ve_server(jid, duong):
    with JOBS_LOCK:
        j = JOBS.get(jid) or {}
        url, token = j.get('callback_url'), j.get('callback_token')
    if not url or not token or not duong or not os.path.exists(duong):
        return None
    mb = os.path.getsize(duong) / 1e6
    t0 = time.perf_counter()
    try:
        with open(duong, 'rb') as f:
            r = requests.post(url, data={'token': token},
                              files={'file': (os.path.basename(duong), f, 'video/mp4')},
                              timeout=(15, 900))
        if r.status_code != 200:
            print(f'[{BACKEND}] day ve server that bai HTTP {r.status_code}: {r.text[:200]}')
            return False
        giay = time.perf_counter() - t0
        print(f'[{BACKEND}] da day {mb:.2f} MB ve server trong {giay:.1f}s '
              f'({mb * 1000 / max(giay, .01):.0f} KB/s)')
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid]['da_day_ve'] = True
        return True
    except Exception as e:
        print(f'[{BACKEND}] day ve server loi: {e}')
        return False


def _run_async(jid, tag, work):
    try:
        with RUN_LOCK:                      # serialize GPU: chi 1 job render 1 luc
            path, filename = work()
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid].update(status='done', stage='done', progress=1.0,
                                 result=path, filename=filename,
                                 so_lieu=dict(LAST_STATS))
        day_ve_server(jid, path)
        print(f'[{tag}] xong job {jid[:8]} -> {path}')
    except Exception as e:
        traceback.print_exc()
        tb = traceback.format_exc()
        frames = [ln for ln in tb.splitlines() if ln.strip().startswith('File "')]
        with JOBS_LOCK:
            if jid in JOBS:
                JOBS[jid].update(status='error', stage='error', error=str(e)[:800],
                                 error_type=type(e).__name__,
                                 where=frames[-1].strip()[:300] if frames else None,
                                 traceback=tb[-2500:])
        print(f'[{tag}] LOI job {jid[:8]}: {type(e).__name__}: {e}')


def _spawn(jid, tag, work, test_case=None):
    with JOBS_LOCK:
        if jid in JOBS:
            JOBS[jid]['test_case'] = _slug(test_case or TEST_CASE)
    print(f'[{tag}] nhan job {jid[:8]} case={_slug(test_case or TEST_CASE)} '
          f'({NOTEBOOK_NAME} {NOTEBOOK_VERSION}) -> chay nen.')
    threading.Thread(target=_run_async, args=(jid, tag, work), daemon=True).start()
    return {'job_id': jid, 'status': 'processing', 'backend': BACKEND,
            'test_case': _slug(test_case or TEST_CASE),
            'result_url': f'/jobs/{jid}/result',
            'notebook': NOTEBOOK_NAME, 'notebook_version': NOTEBOOK_VERSION}


def _active_jobs():
    with JOBS_LOCK:
        return sum(1 for v in JOBS.values() if v.get('status') == 'processing')


def _out_paths(uid, test_case, ext='mp4'):
    fn = f'{uid}_{BACKEND}_{_slug(test_case)}_{NOTEBOOK_VERSION}.{ext}'
    return os.path.join(RES_DIR, fn), fn


STARTED_AT = time.time()


def _json_an_toan(o):
    """Starlette dump JSON voi allow_nan=False -- mot NaN lot vao la endpoint tra HTTP 500."""
    import math
    if isinstance(o, float):
        return o if math.isfinite(o) else str(o)
    if isinstance(o, dict):
        return {k: _json_an_toan(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_json_an_toan(v) for v in o]
    return o


# ============================================================ endpoints
@app.get('/version')
def version():
    try:
        import torch as _t
        gpu = _t.cuda.get_device_name(0) if _t.cuda.is_available() else None
    except Exception:
        gpu = None
    return {
        'notebook': NOTEBOOK_NAME, 'notebook_version': NOTEBOOK_VERSION, 'backend': BACKEND,
        'ai_type': AI_TYPE, 'generate_paths': DUONG_GENERATE, 'test_case': TEST_CASE,
        'models': MODELS, 'changelog': CHANGELOG, 'params': list(BACKEND_PARAMS),
        'mac_dinh': MAC_DINH, 'gpu': gpu, 'models_ready': STATE['models_ready'],
        'uptime_s': round(time.time() - STARTED_AT, 1),
        'jobs': {'total': len(JOBS), 'active': _active_jobs(),
                 'done': sum(1 for v in JOBS.values() if v.get('status') == 'done'),
                 'error': sum(1 for v in JOBS.values() if v.get('status') == 'error')},
    }


@app.get('/health')
def health():
    return {'ok': True, 'notebook': NOTEBOOK_NAME, 'notebook_version': NOTEBOOK_VERSION,
            'backend': BACKEND, 'ai_type': AI_TYPE, 'test_case': TEST_CASE,
            'generate_paths': DUONG_GENERATE,
            'models_ready': STATE['models_ready'], 'active_jobs': _active_jobs(),
            'params': list(BACKEND_PARAMS)}


@app.get('/jobs/{job_id}')
def job_status(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(status_code=404, detail='job not found')
        return _json_an_toan({
            'job_id': job_id, 'status': j['status'], 'stage': j['stage'],
            'progress': j['progress'], 'error': j['error'],
            'elapsed': round(time.time() - j['created'], 1),
            'backend': BACKEND, 'notebook_version': NOTEBOOK_VERSION,
            'test_case': j.get('test_case'), 'filename': j.get('filename'),
            'so_lieu': j.get('so_lieu'),
            'error_type': j.get('error_type'), 'where': j.get('where'),
            'traceback': j.get('traceback')})


@app.get('/jobs/{job_id}/result')
def job_result(job_id: str):
    with JOBS_LOCK:
        j = JOBS.get(job_id)
        if not j:
            raise HTTPException(status_code=404, detail='job not found')
        status, path, fn, err = j['status'], j['result'], j['filename'], j['error']
    if status == 'processing':
        raise HTTPException(status_code=409, detail='job chua xong')
    if status == 'error':
        raise HTTPException(status_code=500, detail=err or 'job error')
    if not path or not os.path.exists(path):
        raise HTTPException(status_code=410, detail='ket qua khong con (da bi don)')
    return FileResponse(path, media_type='video/mp4' if fn.endswith('.mp4') else 'image/png',
                        filename=fn)


@app.get('/laststats')
def laststats():
    return _json_an_toan(LAST_STATS)


@app.get('/', response_class=HTMLResponse)
def trang_chu():
    return (f'<!doctype html><meta charset=utf-8><title>{NOTEBOOK_NAME}</title>'
            f'<body style="font:14px system-ui;max-width:640px;margin:2rem auto">'
            f'<h3>{NOTEBOOK_NAME} {NOTEBOOK_VERSION} &mdash; backend <code>{BACKEND}</code></h3>'
            f'<p>type <code>{AI_TYPE}</code> &middot; <a href=/version>/version</a> &middot; '
            f'<a href=/health>/health</a> &middot; <a href=/laststats>/laststats</a></p>'
            f'<p>POST JSON toi <code>{DUONG_GENERATE[0]}</code>, roi GET /jobs/{{id}} va '
            f'/jobs/{{id}}/result.</p>')


def nhan_job(job: Job):
    uid = uuid.uuid4().hex[:8]
    case = job.test_case or TEST_CASE
    out_path, fn = _out_paths(uid, case, ext='mp4')
    jid = _new_job(job.job_id, job.callback_url, job.callback_token)
    prog = _progress_cb(jid)

    def work():
        return chay_job(job, uid, out_path, progress=prog), fn

    return _spawn(jid, BACKEND, work, case)


# ── Đường mà api.downloadvideo.vn gọi ───────────────────────────────────────
# api dựng đường dẫn THẲNG từ tên type: POST /generate/<type>. Đăng ký CÙNG MỘT hàm
# dưới mọi tên; dict.fromkeys để giữ thứ tự và bỏ trùng.
DUONG_GENERATE = ['/generate/' + t for t in dict.fromkeys([AI_TYPE] + AI_TYPE_ALIAS)]
for _d in DUONG_GENERATE:
    app.post(_d)(nhan_job)


# ==== KHOI DONG ====  (test dung o day: phia duoi can GPU, mang, Colab)
print(f'>>> [{NOTEBOOK_NAME} {NOTEBOOK_VERSION}] backend={BACKEND} case={TEST_CASE}')
_ok = True
for _name, _fn in PRELOAD:
    _t0 = time.time()
    try:
        _fn()
        print(f'   [preload] {_name} OK ({time.time() - _t0:.0f}s)')
    except Exception as _e:
        _ok = False
        traceback.print_exc()
        print(f'   [preload] {_name} LOI: {_e}')
STATE['models_ready'] = _ok
print('>>> MODEL SAN SANG.' if _ok else '>>> MODEL CHUA SAN SANG (xem log tren).')


def _port_in_use(p):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', p)) == 0


if _port_in_use(PORT):
    print(f'uvicorn da chay san o port {PORT} (chay lai cell) -> KHONG khoi dong lai.')
else:
    threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=PORT,
                                                log_level='warning'), daemon=True).start()
    time.sleep(3)

try:
    subprocess.run(['pkill', '-f', 'cloudflared tunnel'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)
except Exception:
    pass

# Bao PUBLIC_URL len api: bot bom window.__ai_boot truoc khi mo notebook.
AI_API_BASE = 'https://api.downloadvideo.vn'


def _read_ai_boot(tries=6, wait=5):
    import json as _json
    try:
        from google.colab import output as _out
    except Exception as e:
        print('   [report] khong co google.colab.output:', e)
        return None
    for i in range(1, tries + 1):
        try:
            raw = _out.eval_js('JSON.stringify(window.__ai_boot || null)', timeout_sec=20)
            boot = _json.loads(raw) if raw and raw != 'null' else None
            if boot and boot.get('account') and boot.get('token'):
                return boot
            print(f'   [report] lan {i}/{tries}: chua thay window.__ai_boot')
        except Exception as e:
            print(f'   [report] lan {i}/{tries}: eval_js loi: {e}')
        time.sleep(wait)
    return None


def _report_public_url(url):
    boot = _read_ai_boot()
    if not boot:
        print('   [report] KHONG co __ai_boot (chay tay?) -> bot se quet dong PUBLIC_URL=.')
        return False
    api = (boot.get('api') or AI_API_BASE).rstrip('/')
    body = {'account': boot['account'], 'token': boot['token'], 'url': url}
    hdr = {'ngrok-skip-browser-warning': '1', 'User-Agent': 'colab-tunnel-report/1.0'}
    for attempt in range(1, 25):
        try:
            r = requests.post(f'{api}/api/c/ai/profiles/tunnel', json=body, headers=hdr, timeout=20)
            print(f'   [report] {attempt}/24 -> {r.status_code} {r.text[:200]}')
            if r.status_code == 200 and r.json().get('success'):
                return True
            if r.status_code in (400, 401):
                return False
        except Exception as e:
            print(f'   [report] {attempt}/24 loi mang: {e}')
        time.sleep(15)
    return False


proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if m and not public_url:
        public_url = m.group(0)
        print('\n\nPUBLIC_URL=' + public_url + '\n', flush=True)
        break

if public_url:
    threading.Thread(target=_report_public_url, args=(public_url,), daemon=True).start()

print(f'=== {NOTEBOOK_NAME} {NOTEBOOK_VERSION} | backend={BACKEND} | case={TEST_CASE} ===')
print('URL:', public_url)
print('Endpoints: GET /version | GET /health | POST ' + DUONG_GENERATE[0] + ' (JSON)')
print('           GET /jobs/{id} | GET /jobs/{id}/result | GET /laststats')
print('>>> SAN SANG. Giu cell nay chay.')
while True:
    line = proc.stdout.readline()
    if not line:
        break
    if 'ERR' in line or 'error' in line.lower():
        print(line, end='')


>>> [ImageToVideo_Wan22_Colab.ipynb v4] backend=wan22_i2v case=W4-81f-480p-Q4-6b-loraT2V
   [wan22_i2v] du 6 file model, GPU: Tesla T4
   [preload] kiem_san_sang OK (0s)
>>> MODEL SAN SANG.
2026-09-17T01:34:26Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-17T01:34:26Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-17T01:34:30Z INF +--------------------------------------------------------------------------------------------+
2026-09-17T

/content/ComfyUI/custom_nodes/ComfyUI_GGUF/loader.py:91: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  torch_tensor = torch.from_numpy(tensor.data) # mmap


   [wan22_i2v] denoise_cao ...
Attempting to release mmap (36)
Patching comfy attention to use sageattn


  0%|          | 0/2 [00:00<?, ?it/s]

Restoring initial comfy attention
   [wan22_i2v] nap_model_thap ...
gguf qtypes: F16 (694), Q4_K (280), Q6_K (120), F32 (1)
   [wan22_i2v] denoise_thap ...
Attempting to release mmap (36)
Patching comfy attention to use sageattn


...

  0%|          | 0/2 [00:00<?, ?it/s]

Restoring initial comfy attention
   [wan22_i2v] giai_ma ...
   [wan22_i2v] ghi_video ...
   [wan22_i2v] xong 65 khung trong 415s -> /content/results/77e74f69_wan22_i2v_W4-A-70s-game-show-host_v4.mp4
[wan22_i2v] xong job w-6 -> /content/results/77e74f69_wan22_i2v_W4-A-70s-game-show-host_v4.mp4
   [wan22_i2v] LoRA t2v: lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank32_bf16.safetensors   (khop kien truc Wan2.2)
   [wan22_i2v] anh 720x1280 -> video 480x848 x 65 khung @ 16 fps, 4 buoc (cao 2 + thap 2), lightx2v 3.0/1.5, cfg 1.0, seed 42
   [wan22_i2v] CHU Y cfg=1.0: negative_prompt KHONG co tac dung. Khong nang cfg duoc — lightx2v la LoRA cfg-step-distill, chay dung o cfg=1; nang cfg la pha chung cat. Muon doi ket qua thi doi chuyen_dong (prompt duong).
   [wan22_i2v] ma_hoa_prompt ...
   [wan22_i2v] prompt da co trong kho -> bo qua nap umt5 (kho 1/16, tiet kiem ~29 s)
   [wan22_i2v] ma_hoa_anh ...
   [wan22_i2v] nap_model_cao ...
gguf qtypes: F16 (694), Q4_K (280), Q6_K (120), F32 (1)
   [

  0%|          | 0/2 [00:00<?, ?it/s]

Restoring initial comfy attention
   [wan22_i2v] nap_model_thap ...
gguf qtypes: F16 (694), Q4_K (280), Q6_K (120), F32 (1)
   [wan22_i2v] denoise_thap ...
Attempting to release mmap (36)
Patching comfy attention to use sageattn


  0%|          | 0/2 [00:00<?, ?it/s]